In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce_gold;

In [0]:
from pyspark.sql.functions import col

# Load Silver tables
users     = spark.table("ecommerce.users_silver")
buyers    = spark.table("ecommerce.buyers_silver")
sellers   = spark.table("ecommerce.sellers_silver")
countries = spark.table("ecommerce.countries_silver")

# Customer 360 Mart
customer360 = (users.join(buyers, "country", "left")
                    .join(countries, "country", "left"))

# Rename overlapping columns
customer360 = customer360.select(
    users["country"].alias("Country"),
    users["productsSold"].alias("Users_productsSold"),
    users["socialnbfollowers"].alias("Users_socialnbfollowers"),
    buyers["buyers"].alias("Buyers_Total"),
    buyers["topbuyers"].alias("Buyers_Top"),
    countries["sellers"].alias("Countries_Sellers"),
    countries["topsellers"].alias("Countries_TopSellers"),
    countries["meanfollowers"].alias("Countries_MeanFollowers"),
    countries["meanfollowing"].alias("Countries_MeanFollowing")
)

customer360.write.format("delta").mode("overwrite").saveAsTable("ecommerce.customer360")

# Seller 360 Mart
seller360 = (sellers.join(countries, "country", "left"))

seller360 = seller360.select(
    sellers["country"].alias("Country"),
    sellers["nbsellers"].alias("Sellers_Total"),
    sellers["meanproductssold"].alias("Sellers_MeanProductsSold"),
    sellers["meansellerpassrate"].alias("Sellers_PassRate"),
    countries["sellers"].alias("Countries_Sellers"),
    countries["topsellers"].alias("Countries_TopSellers"),
    countries["meanfollowers"].alias("Countries_MeanFollowers"),
    countries["meanfollowing"].alias("Countries_MeanFollowing")
)

seller360.write.format("delta").mode("overwrite").saveAsTable("ecommerce.seller360")

# Market Insights Mart
market_insights = (buyers.join(sellers, "country", "left")
                          .join(countries, "country", "left"))

market_insights = market_insights.select(
    buyers["country"].alias("Country"),
    buyers["buyers"].alias("Buyers_Total"),
    buyers["topbuyers"].alias("Buyers_Top"),
    sellers["nbsellers"].alias("Sellers_Total"),
    sellers["meanproductssold"].alias("Sellers_MeanProductsSold"),
    countries["sellers"].alias("Countries_Sellers"),
    countries["topsellers"].alias("Countries_TopSellers"),
    countries["meanfollowers"].alias("Countries_MeanFollowers"),
    countries["meanfollowing"].alias("Countries_MeanFollowing")
)

market_insights.write.format("delta").mode("overwrite").saveAsTable("ecommerce.market_insights")
